In [6]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(str(Path.cwd().parent))

from data.loader import load_tables, load_excel_files
from data.calculations import (
    build_product_classification,
    build_product_families,
    enrich_sales_with_classification,
    build_customer_activity,
    enrich_customers_with_activity,
    enrich_sales_with_customer_activity,
    filter_sales_by_bodega,
    prepare_sales
)

# -----------------------------
# Load raw data
# -----------------------------
access_file = "/Users/ricardolugo/Library/CloudStorage/OneDrive-Personal/LH/Reports/sales_lh.accdb"
tables = ["sales", "customers"]

data = load_tables(access_file, tables)

df_sales = data["sales"]
df_customers = data["customers"]
df_actividades, df_clasificacion, df_inventory, df_crm = load_excel_files()

# -----------------------------
# Build reference tables
# -----------------------------
df_grupos = build_product_classification(df_clasificacion)
df_familias = build_product_families(df_clasificacion)
df_customer_activity = build_customer_activity(df_actividades)

# -----------------------------
# Enrich tables
# -----------------------------
df_sales_enriched = enrich_sales_with_classification(
    df_sales,
    df_grupos,
    df_familias
)

df_customers_enriched = enrich_customers_with_activity(
    df_customers,
    df_customer_activity
)

df_sales_final = enrich_sales_with_customer_activity(
    df_sales_enriched,
    df_customers_enriched
)

# -----------------------------
# Working filter: Cali
# -----------------------------

df_sales_clean = prepare_sales(df_sales_final)

df_cali = filter_sales_by_bodega(df_sales_clean, 50)
df_bogota = filter_sales_by_bodega(df_sales_clean, 1)

Loading sales...
sales loaded: (285619, 25)
Loading customers...
customers loaded: (36429, 16)
Actividades shape: (594, 6)
Clasificacion shape: (203, 4)
Inventario shape: (11169, 32)
CRM shape: (539, 16)
Cotizacion shape: (374, 12)


ValueError: too many values to unpack (expected 4)

In [ ]:
# -----------------------------
# BLOQUE 1: TAMANO DEL NEGOCIO CALI
# -----------------------------

import pandas as pd

# Asegurar tipos

df_cali_analysis = df_cali.copy()

df_cali_analysis["fecha"] = pd.to_datetime(df_cali_analysis["fecha"], errors="coerce")
df_cali_analysis["valorbruto"] = pd.to_numeric(df_cali_analysis["valorbruto"], errors="coerce")
df_cali_analysis["cantidad"] = pd.to_numeric(df_cali_analysis["cantidad"], errors="coerce")
df_cali_analysis["precio"] = pd.to_numeric(df_cali_analysis["precio"], errors="coerce")
df_cali_analysis["costo"] = pd.to_numeric(df_cali_analysis["costo"], errors="coerce")

# Helpers
df_cali_analysis["year"] = df_cali_analysis["fecha"].dt.year
df_cali_analysis["year_month"] = df_cali_analysis["fecha"].dt.to_period("M")

In [ ]:
df_cali_analysis["ventas_mm"] = df_cali_analysis["valorbruto"] / 1_000_000

# -----------------------------
# RUN RATE (ULTIMOS MESES)
# -----------------------------

today = pd.Timestamp("2026-04-01")

df_cali_analysis["fecha"] = pd.to_datetime(
    df_cali_analysis["fecha"],
    format="%d/%m/%y %H:%M:%S",
    errors="coerce"
)

#df_cali_analysis["ventas_mm"] = pd.to_numeric(df_cali_analysis["valorbruto"], errors="coerce")

# últimos 3 meses
df_last_3m = df_cali_analysis[
    df_cali_analysis["fecha"] >= today - pd.DateOffset(months=3)
]

# últimos 6 meses
df_last_6m = df_cali_analysis[
    df_cali_analysis["fecha"] >= today - pd.DateOffset(months=6)
]

print("Ventas últimos 3 meses:", f'{df_last_3m["ventas_mm"].sum():,.0f}')
print("Ventas últimos 6 meses:", f'{df_last_6m["ventas_mm"].sum():,.0f}')

Ventas últimos 3 meses: 1,331
Ventas últimos 6 meses: 2,720


In [ ]:
# ---------------------------------------------------------
# tipos
# ---------------------------------------------------------

df_cali_analysis["fecha"] = pd.to_datetime(
    df_cali_analysis["fecha"],
    errors="coerce"
)

numeric_cols = [
    "valorbruto",
    "cantidad",
    "precio",
    "costo",
]

for col in numeric_cols:
    df_cali_analysis[col] = pd.to_numeric(
        df_cali_analysis[col],
        errors="coerce"
    ).fillna(0)

# si cantidad viene vacía o <= 0
df_cali_analysis.loc[
    df_cali_analysis["cantidad"] <= 0,
    "cantidad"
] = 1

# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------

df_cali_analysis["year"] = (
    df_cali_analysis["fecha"].dt.year
)

df_cali_analysis["year_month"] = (
    df_cali_analysis["fecha"].dt.to_period("M")
)

# =========================================================
# VARIABLES REALES DE NEGOCIO
# =========================================================

# venta total línea
df_cali_analysis["venta_total"] = (
    df_cali_analysis["valorbruto"]
)

# costo total línea
df_cali_analysis["costo_total"] = (
    df_cali_analysis["costo"] *
    df_cali_analysis["cantidad"]
)

# utilidad real
df_cali_analysis["utilidad_total"] = (
    df_cali_analysis["venta_total"] -
    df_cali_analysis["costo_total"]
)

# margen %
df_cali_analysis["margen_pct"] = np.where(
    df_cali_analysis["venta_total"] > 0,
    df_cali_analysis["utilidad_total"] /
    df_cali_analysis["venta_total"],
    0
)

# =========================================================
# VERSION EN MILLONES COP
# =========================================================

df_cali_analysis["ventas_mm"] = (
    df_cali_analysis["venta_total"] / 1_000_000
)

df_cali_analysis["costos_mm"] = (
    df_cali_analysis["costo_total"] / 1_000_000
)

df_cali_analysis["utilidad_mm"] = (
    df_cali_analysis["utilidad_total"] / 1_000_000
)

# =========================================================
# VALIDACION RAPIDA
# =========================================================

print(
    df_cali_analysis[
        [
            "razonsocial",
            "sku",
            "cantidad",
            "precio",
            "valorbruto",
            "venta_total",
            "costo",
            "costo_total",
            "utilidad_total",
            "margen_pct",
        ]
    ].head(10)
)

                  razonsocial                   sku  cantidad   precio  \
249      INDUSTRIAS JILA LTDA  6007-C-2Z-L038-C3FAG       1.0  20400.0   
250      INDUSTRIAS JILA LTDA   K 21X25X13-A/0-7INA       1.0  22600.0   
251      INDUSTRIAS JILA LTDA        NK 20/20-XLINA       8.0  32900.0   
252  ALMACEN RODAMIENTOS S.A.        6203-2Z/GJNSKF       8.0  12500.0   
253      INDUSTRIAS JILA LTDA              51105NTN       1.0  22300.0   
254      INDUSTRIAS JILA LTDA   6006-C-2HRS-L038FAG       1.0  19100.0   
255      INDUSTRIAS JILA LTDA  20X42X7 HMSA10 RGSKF       2.0   8000.0   
256      INDUSTRIAS JILA LTDA        6004-2Z/GJNSKF       2.0  15100.0   
257      INDUSTRIAS JILA LTDA            628 LLBNTN       1.0  10700.0   
258  ALMACEN RODAMIENTOS S.A.       E-5 MANZANASREX       1.0  85300.0   

     valorbruto  venta_total         costo    costo_total  utilidad_total  \
249     20400.0      20400.0  14526.529674   14526.529674     5873.470326   
250     22600.0      22600.0   

In [ ]:
# =========================================================
# RESUMEN CLIENTES CALI - VERSION CORRECTA
# =========================================================

max_date = df_cali_analysis["fecha"].max()

start_12m = max_date - pd.DateOffset(months=12)
start_6m = max_date - pd.DateOffset(months=6)
start_prev_12m = start_12m - pd.DateOffset(months=12)

df_12m = df_cali_analysis[df_cali_analysis["fecha"] >= start_12m].copy()
df_6m = df_cali_analysis[df_cali_analysis["fecha"] >= start_6m].copy()
df_prev_12m = df_cali_analysis[
    (df_cali_analysis["fecha"] >= start_prev_12m) &
    (df_cali_analysis["fecha"] < start_12m)
].copy()

# base histórica
base = (
    df_cali_analysis
    .groupby(["nit", "razonsocial"], as_index=False)
    .agg(
        primera_compra=("fecha", "min"),
        ultima_compra=("fecha", "max"),
        ventas_historicas=("venta_total", "sum"),
        utilidad_historica=("utilidad_total", "sum"),
        facturas_historicas=("numero", "nunique"),
        skus_historicos=("sku", "nunique"),
        actividad_economica=("actividad_economica", "last"),
    )
)

# últimos 12 meses
agg_12m = (
    df_12m
    .groupby(["nit", "razonsocial"], as_index=False)
    .agg(
        ventas_12m=("venta_total", "sum"),
        utilidad_12m=("utilidad_total", "sum"),
        facturas_12m=("numero", "nunique"),
        skus_12m=("sku", "nunique"),
        ultima_compra_12m=("fecha", "max"),
    )
)

# últimos 6 meses
agg_6m = (
    df_6m
    .groupby(["nit", "razonsocial"], as_index=False)
    .agg(
        ventas_6m=("venta_total", "sum"),
        utilidad_6m=("utilidad_total", "sum"),
        facturas_6m=("numero", "nunique"),
        skus_6m=("sku", "nunique"),
    )
)

# 12 meses previos
agg_prev_12m = (
    df_prev_12m
    .groupby(["nit", "razonsocial"], as_index=False)
    .agg(
        ventas_prev_12m=("venta_total", "sum"),
        utilidad_prev_12m=("utilidad_total", "sum"),
        facturas_prev_12m=("numero", "nunique"),
    )
)

# merge
resumen_clientes = (
    base
    .merge(agg_12m, on=["nit", "razonsocial"], how="left")
    .merge(agg_6m, on=["nit", "razonsocial"], how="left")
    .merge(agg_prev_12m, on=["nit", "razonsocial"], how="left")
)

cols_fill_zero = [
    "ventas_12m", "utilidad_12m", "facturas_12m", "skus_12m",
    "ventas_6m", "utilidad_6m", "facturas_6m", "skus_6m",
    "ventas_prev_12m", "utilidad_prev_12m", "facturas_prev_12m"
]

for col in cols_fill_zero:
    resumen_clientes[col] = resumen_clientes[col].fillna(0)

# métricas
resumen_clientes["dias_sin_comprar"] = (
    max_date - resumen_clientes["ultima_compra"]
).dt.days

resumen_clientes["ticket_promedio_12m"] = np.where(
    resumen_clientes["facturas_12m"] > 0,
    resumen_clientes["ventas_12m"] / resumen_clientes["facturas_12m"],
    0
)

resumen_clientes["margen_bruto_pct_12m"] = np.where(
    resumen_clientes["ventas_12m"] > 0,
    resumen_clientes["utilidad_12m"] / resumen_clientes["ventas_12m"],
    0
)

resumen_clientes["variacion_ventas_12m_vs_prev"] = np.where(
    resumen_clientes["ventas_prev_12m"] > 0,
    (resumen_clientes["ventas_12m"] / resumen_clientes["ventas_prev_12m"]) - 1,
    np.nan
)

# estado comercial
conditions = [
    (resumen_clientes["ventas_12m"] > 0) & (resumen_clientes["dias_sin_comprar"] <= 60),
    (resumen_clientes["ventas_12m"] > 0) & (resumen_clientes["dias_sin_comprar"].between(61, 120)),
    (resumen_clientes["ventas_12m"] > 0) & (resumen_clientes["dias_sin_comprar"] > 120),
    (resumen_clientes["ventas_12m"] == 0) & (resumen_clientes["ventas_historicas"] > 0),
]

choices = [
    "Activa",
    "En Riesgo",
    "Dormida",
    "Sin compra reciente",
]

resumen_clientes["estado_comercial"] = np.select(
    conditions,
    choices,
    default="Nueva / sin historia reciente"
)

# ranking
resumen_clientes = (
    resumen_clientes
    .sort_values(["ventas_12m", "utilidad_12m"], ascending=[False, False])
    .reset_index(drop=True)
)

resumen_clientes["rank_ventas_12m"] = resumen_clientes.index + 1

In [ ]:
top_cuentas_cali = resumen_clientes.copy()

top_cuentas_cali = top_cuentas_cali[
    ~top_cuentas_cali["actividad_economica"].isin([
        "COMERCIO EN GENERAL",
        "COMERCIO DE RODAMIENTOS Y AFINES",
    ])
].copy()

cols_millones = [
    "ventas_12m",
    "ventas_6m",
    "utilidad_12m",
    "ticket_promedio_12m",
]

for col in cols_millones:
    top_cuentas_cali[col] = (top_cuentas_cali[col] / 1_000_000).round(1)

top_cuentas_cali["margen_bruto_pct_12m"] = (
    top_cuentas_cali["margen_bruto_pct_12m"] * 100
).round(1)

top_cuentas_cali["variacion_ventas_12m_vs_prev"] = (
    top_cuentas_cali["variacion_ventas_12m_vs_prev"] * 100
).round(1)

top_cuentas_cali = top_cuentas_cali[
    [
        "rank_ventas_12m",
        "razonsocial",
        "actividad_economica",
        "ventas_12m",
        "ventas_6m",
        "utilidad_12m",
        "margen_bruto_pct_12m",
        "facturas_12m",
        "skus_12m",
        "ticket_promedio_12m",
        "ultima_compra",
        "dias_sin_comprar",
        "variacion_ventas_12m_vs_prev",
        "estado_comercial",
    ]
].reset_index(drop=True)

top_cuentas_cali.head(30)

,rank_ventas_12m,razonsocial,actividad_economica,ventas_12m,ventas_6m,utilidad_12m,margen_bruto_pct_12m,facturas_12m,skus_12m,ticket_promedio_12m,ultima_compra,dias_sin_comprar,variacion_ventas_12m_vs_prev,estado_comercial
0,1,CARTONES AMERICA S.A.,FABRICACION PRODUCTOS CARTON Y PAPEL,557.2,284.2,140.7,25.3,309.0,352.0,1.8,2026-03-25,6,-14.8,Activa
1,3,SERMOTOR INGENIERIA S.A.S,TALLERES,215.7,106.9,64.5,29.9,123.0,280.0,1.8,2026-03-31,0,75.8,Activa
2,5,IME INGENIERIA DE MAQUINAS ELECTRICAS S.A.S,TALLERES,135.9,65.1,46.4,34.1,105.0,120.0,1.3,2026-03-27,4,111.2,Activa
3,6,NESTLE DE COLOMBIA S.A.,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,133.1,52.0,49.9,37.5,43.0,228.0,3.1,2026-03-12,19,14.0,Activa
4,7,PAPELES NACIONALES S.A.S,FABRICACION PRODUCTOS CARTON Y PAPEL,131.7,38.5,25.1,19.0,106.0,196.0,1.2,2026-03-20,11,-19.5,Activa
5,9,CARTON DE COLOMBIA S.A.,FABRICACION PULPA DE PAPEL,121.6,97.2,76.3,62.7,50.0,48.0,2.4,2026-03-27,4,207.1,Activa
6,10,PGI COLOMBIA LTDA.,PRODUCTOS TEXTILES,118.5,76.7,38.4,32.4,44.0,78.0,2.7,2026-03-19,12,13.4,Activa
7,11,POLLOS EL BUCANERO S.A.,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,106.6,55.9,31.6,29.7,50.0,76.0,2.1,2026-03-17,14,212.8,Activa
8,12,REFINADORA NACIONAL DE ACEITES Y GRASAS S.A.S,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,103.9,42.8,41.5,39.9,70.0,85.0,1.5,2026-03-25,6,167.8,Activa
9,13,CENTRAL DE REBOBINADOS MOTORES ELECTRICOS S.A.S.,TALLERES,103.7,45.2,34.1,32.9,53.0,74.0,2.0,2026-03-20,11,-8.2,Activa


In [ ]:
df_customers_enriched.columns

Index(['nit', 'dv', 'razonsocial', 'escliente', 'cliente_credito',
       'direccion1', 'ciudad', 'telefono1', 'movil', 'email', 'emailfe',
       'vendedor', 'cupocreditocc', 'plazopagocc', 'idciiu', 'id',
       'actividad_economica'],
      dtype='object')

In [ ]:
# =========================================================
# AGREGAR VENDEDOR AL RESUMEN DE CLIENTES
# =========================================================

# tomar maestro único de clientes
df_vendedores = (
    df_customers_enriched[
        [
            "nit",
            "razonsocial",
            "vendedor",
            "ciudad",
            "actividad_economica",
        ]
    ]
    .drop_duplicates(subset=["nit"])
    .copy()
)

# merge contra resumen_clientes
resumen_clientes = resumen_clientes.merge(
    df_vendedores[
        [
            "nit",
            "vendedor",
            "ciudad",
        ]
    ],
    on="nit",
    how="left"
)

# =========================================================
# REORDENAR COLUMNAS IMPORTANTES
# =========================================================

cols_final = [
    "rank_ventas_12m",
    "razonsocial",
    "vendedor",
    "ciudad",
    "actividad_economica",

    "ventas_12m",
    "ventas_6m",
    "utilidad_12m",
    "margen_bruto_pct_12m",

    "facturas_12m",
    "skus_12m",
    "ticket_promedio_12m",

    "ultima_compra",
    "dias_sin_comprar",

    "variacion_ventas_12m_vs_prev",
    "estado_comercial",
]

top_cuentas_cali = (
    resumen_clientes[cols_final]
    .copy()
)

top_cuentas_cali.head(30)

,rank_ventas_12m,razonsocial,vendedor,ciudad,actividad_economica,ventas_12m,ventas_6m,utilidad_12m,margen_bruto_pct_12m,facturas_12m,skus_12m,ticket_promedio_12m,ultima_compra,dias_sin_comprar,variacion_ventas_12m_vs_prev,estado_comercial
0,1,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,MOSQUERA,FABRICACION PRODUCTOS CARTON Y PAPEL,5.571873e+08,2.841539e+08,1.407021e+08,0.252522,309.0,352.0,1.803195e+06,2026-03-25,6,-0.148430,Activa
1,2,SANCHEZ Y ESCOBAR RODAMIENTOS S.A.S.,ALMACEN CALI -UNO-,CALI,COMERCIO EN GENERAL,2.904617e+08,1.288901e+08,1.025811e+08,0.353166,402.0,836.0,7.225416e+05,2026-03-31,0,-0.319752,Activa
2,3,SERMOTOR INGENIERIA S.A.S,FABIO NELSON VALENCIA,CALI,TALLERES,2.157416e+08,1.069053e+08,6.451038e+07,0.299017,123.0,280.0,1.753997e+06,2026-03-31,0,0.757568,Activa
3,4,RODICLAR S.A.S.,ALMACEN CALI -UNO-,CALI,COMERCIO DE RODAMIENTOS Y AFINES,1.696776e+08,7.496083e+07,6.495937e+07,0.382840,335.0,350.0,5.065004e+05,2026-03-31,0,0.048669,Activa
4,5,IME INGENIERIA DE MAQUINAS ELECTRICAS S.A.S,JAIRO DAVID VERA,CALI,TALLERES,1.359207e+08,6.505184e+07,4.638086e+07,0.341235,105.0,120.0,1.294483e+06,2026-03-27,4,1.111571,Activa
5,6,NESTLE DE COLOMBIA S.A.,JEISMAN HOLGUIN,BUGALAGRANDE,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,1.330935e+08,5.196594e+07,4.992634e+07,0.375122,43.0,228.0,3.095197e+06,2026-03-12,19,0.140033,Activa
6,7,PAPELES NACIONALES S.A.S,JAIRO DAVID VERA,PEREIRA,FABRICACION PRODUCTOS CARTON Y PAPEL,1.317378e+08,3.845926e+07,2.507675e+07,0.190354,106.0,196.0,1.242809e+06,2026-03-20,11,-0.195025,Activa
7,8,ALMACEN RODAMIENTOS S.A.,ALMACEN,BOGOTA D.C.,COMERCIO EN GENERAL,1.236497e+08,4.606994e+07,4.500530e+07,0.363974,184.0,159.0,6.720092e+05,2026-03-20,11,0.115183,Activa
8,9,CARTON DE COLOMBIA S.A.,NUBIA ANDREA JIMENEZ,YUMBO,FABRICACION PULPA DE PAPEL,1.216071e+08,9.723164e+07,7.628185e+07,0.627281,50.0,48.0,2.432142e+06,2026-03-27,4,2.071385,Activa
9,10,PGI COLOMBIA LTDA.,JEISMAN HOLGUIN,PALMIRA,PRODUCTOS TEXTILES,1.185368e+08,7.668815e+07,3.841854e+07,0.324106,44.0,78.0,2.694018e+06,2026-03-19,12,0.133760,Activa


In [ ]:
# =========================================================
# PRIORIDAD COMERCIAL AUTOMATICA
# =========================================================

import numpy as np

scoring = resumen_clientes.copy()

# ---------------------------------------------------------
# helper min-max normalization
# ---------------------------------------------------------

def minmax(series):
    series = series.fillna(0)

    if series.max() == series.min():
        return pd.Series(0, index=series.index)

    return (
        (series - series.min()) /
        (series.max() - series.min())
    )

# ---------------------------------------------------------
# componentes del score
# ---------------------------------------------------------

# tamaño de cuenta
scoring["score_ventas"] = minmax(
    scoring["ventas_12m"]
)

# rentabilidad
scoring["score_utilidad"] = minmax(
    scoring["utilidad_12m"]
)

# recencia (más alto = más días sin comprar = más riesgo)
scoring["score_recencia"] = minmax(
    scoring["dias_sin_comprar"]
)

# caída comercial
# solo castiga caídas, no premia crecimientos exagerados
scoring["score_caida"] = minmax(
    scoring["variacion_ventas_12m_vs_prev"]
    .fillna(0)
    .apply(lambda x: abs(x) if x < 0 else 0)
)

# ---------------------------------------------------------
# score final
# ---------------------------------------------------------

scoring["priority_score"] = (
    scoring["score_ventas"] * 0.35 +
    scoring["score_utilidad"] * 0.20 +
    scoring["score_recencia"] * 0.25 +
    scoring["score_caida"] * 0.20
)

# ---------------------------------------------------------
# clasificación gerencial
# ---------------------------------------------------------

conditions = [
    scoring["priority_score"] >= 0.75,
    scoring["priority_score"] >= 0.55,
    scoring["priority_score"] >= 0.35,
]

choices = [
    "Critica",
    "Alta",
    "Media",
]

scoring["prioridad_comercial"] = np.select(
    conditions,
    choices,
    default="Baja"
)

# ---------------------------------------------------------
# ranking
# ---------------------------------------------------------

scoring = (
    scoring
    .sort_values(
        "priority_score",
        ascending=False
    )
    .reset_index(drop=True)
)

scoring["rank_prioridad"] = (
    scoring.index + 1
)

# ---------------------------------------------------------
# salida final
# ---------------------------------------------------------

top_prioridad = scoring[
    [
        "rank_prioridad",
        "razonsocial",
        "vendedor",
        "actividad_economica",

        "ventas_12m",
        "utilidad_12m",
        "dias_sin_comprar",

        "variacion_ventas_12m_vs_prev",
        "estado_comercial",

        "priority_score",
        "prioridad_comercial",
    ]
].copy()

# mostrar ventas en millones
cols_money = [
    "ventas_12m",
    "utilidad_12m",
]

for col in cols_money:
    top_prioridad[col] = (
        top_prioridad[col] / 1_000_000
    ).round(1)

top_prioridad["variacion_ventas_12m_vs_prev"] = (
    top_prioridad["variacion_ventas_12m_vs_prev"] * 100
).round(1)

top_prioridad["priority_score"] = (
    top_prioridad["priority_score"]
).round(2)

top_prioridad.head(30)

,rank_prioridad,razonsocial,vendedor,actividad_economica,ventas_12m,utilidad_12m,dias_sin_comprar,variacion_ventas_12m_vs_prev,estado_comercial,priority_score,prioridad_comercial
0,1,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,FABRICACION PRODUCTOS CARTON Y PAPEL,557.2,140.7,6,-14.8,Activa,0.58,Alta
1,2,SANCHEZ Y ESCOBAR RODAMIENTOS S.A.S.,ALMACEN CALI -UNO-,COMERCIO EN GENERAL,290.5,102.6,0,-32.0,Activa,0.40,Media
2,3,HUMUS ABONOS ORGANICOS Y SOLUCIONES ECOLOGICAS...,ALMACEN CALI -UNO-,OTRAS ACTIVIDADES EMPRESARIALES,0.0,0.0,729,-100.0,Sin compra reciente,0.37,Media
3,4,METALVIAL S.A.S.,ALMACEN CALI -UNO-,COMERCIO MATERIALES DE CONSTRUCCION,0.0,0.0,728,-100.0,Sin compra reciente,0.37,Media
4,5,VALENCIA DIAZ JUAN FERNANDO,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,0.0,0.0,728,-100.0,Sin compra reciente,0.37,Media
5,6,TECNIRODAMIENTOS LTDA,ALMACEN,COMERCIO Y MANTENIMIENTO DE AUTOMOTORES,0.0,0.0,728,-100.0,Sin compra reciente,0.37,Media
6,7,GREENPULP S.A.S,ALMACEN CALI -UNO-,FABRICACION PULPA DE PAPEL,0.0,0.0,727,-100.0,Sin compra reciente,0.37,Media
7,8,CHAVESTAN LONDONO ARACELLY,WHATSAPP CALI,COMERCIO DE RODAMIENTOS Y AFINES,0.0,0.0,727,-100.0,Sin compra reciente,0.37,Media
8,9,COBO VALDERRAMA RONALD ALEXIS,ALMACEN CALI -UNO-,COMERCIO DE RODAMIENTOS Y AFINES,0.0,0.0,726,-100.0,Sin compra reciente,0.37,Media
9,10,GRIJALVA VICTOR,ALMACEN CALI -UNO-,INDUSTRIAS METALURGICAS,0.0,0.0,726,-100.0,Sin compra reciente,0.37,Media


In [ ]:
# =========================================================
# CUSTOMER DEEP DIVE
# ejemplo inicial:
# CARTONES AMERICA S.A.
# =========================================================

import pandas as pd
import numpy as np

cliente = "CARTONES AMERICA S.A."

df_cliente = (
    df_cali_analysis[
        df_cali_analysis["razonsocial"] == cliente
    ]
    .copy()
)

# ---------------------------------------------------------
# validación
# ---------------------------------------------------------

print("Cliente:", cliente)
print("Rows:", df_cliente.shape[0])

# =========================================================
# RESUMEN GENERAL
# =========================================================

resumen_cliente = {
    "cliente": cliente,
    "ventas_totales_mm": round(
        df_cliente["ventas_mm"].sum(), 1
    ),
    "utilidad_total_mm": round(
        df_cliente["utilidad_mm"].sum(), 1
    ),
    "facturas": int(
        df_cliente["numero"].nunique()
    ),
    "skus": int(
        df_cliente["sku"].nunique()
    ),
    "ultima_compra": df_cliente["fecha"].max(),
    "primera_compra": df_cliente["fecha"].min(),
}

print("\nRESUMEN GENERAL")
print(resumen_cliente)

# =========================================================
# TENDENCIA MENSUAL
# =========================================================

monthly_trend = (
    df_cliente
    .groupby("year_month", as_index=False)
    .agg(
        ventas_mm=("ventas_mm", "sum"),
        utilidad_mm=("utilidad_mm", "sum"),
        facturas=("numero", "nunique"),
        skus=("sku", "nunique"),
    )
    .sort_values("year_month")
)

print("\nTENDENCIA MENSUAL")
print(monthly_trend.tail(12))

# =========================================================
# TOP FAMILIAS
# =========================================================

top_familias = (
    df_cliente
    .groupby(
        ["nombre_familia"],
        as_index=False
    )
    .agg(
        ventas_mm=("ventas_mm", "sum"),
        utilidad_mm=("utilidad_mm", "sum"),
        skus=("sku", "nunique"),
        facturas=("numero", "nunique"),
    )
    .sort_values(
        "ventas_mm",
        ascending=False
    )
)

print("\nTOP FAMILIAS")
print(top_familias.head(15))

# =========================================================
# TOP SKUs
# =========================================================

top_skus = (
    df_cliente
    .groupby(
        ["sku", "nombreproducto"],
        as_index=False
    )
    .agg(
        ventas_mm=("ventas_mm", "sum"),
        utilidad_mm=("utilidad_mm", "sum"),
        cantidad_total=("cantidad", "sum"),
        facturas=("numero", "nunique"),
    )
    .sort_values(
        "ventas_mm",
        ascending=False
    )
)

print("\nTOP SKUS")
print(top_skus.head(20))

# =========================================================
# TOP MARCAS
# =========================================================

top_marcas = (
    df_cliente
    .groupby(
        ["sufijo"],
        as_index=False
    )
    .agg(
        ventas_mm=("ventas_mm", "sum"),
        utilidad_mm=("utilidad_mm", "sum"),
        skus=("sku", "nunique"),
    )
    .sort_values(
        "ventas_mm",
        ascending=False
    )
)

print("\nTOP MARCAS")
print(top_marcas.head(10))

# =========================================================
# ULTIMAS COMPRAS
# =========================================================

ultimas_compras = (
    df_cliente
    .sort_values(
        "fecha",
        ascending=False
    )[
        [
            "fecha",
            "numero",
            "sku",
            "nombreproducto",
            "cantidad",
            "ventas_mm",
            "utilidad_mm",
        ]
    ]
    .head(20)
)

print("\nULTIMAS COMPRAS")
print(ultimas_compras)

# =========================================================
# YOY SIMPLE
# ventas por año
# =========================================================

ventas_yoy = (
    df_cliente
    .groupby(
        "year",
        as_index=False
    )
    .agg(
        ventas_mm=("ventas_mm", "sum"),
        utilidad_mm=("utilidad_mm", "sum"),
        facturas=("numero", "nunique"),
    )
    .sort_values("year")
)

print("\nVENTAS POR AÑO")
ventas_yoy

Cliente: CARTONES AMERICA S.A.
Rows: 1631

RESUMEN GENERAL
{'cliente': 'CARTONES AMERICA S.A.', 'ventas_totales_mm': 1713.7, 'utilidad_total_mm': 257.5, 'facturas': 796, 'skus': 685, 'ultima_compra': Timestamp('2026-03-25 00:00:00'), 'primera_compra': Timestamp('2023-01-04 00:00:00')}

TENDENCIA MENSUAL
   year_month  ventas_mm  utilidad_mm  facturas  skus
27    2025-04  97.416322    17.001173        44    71
28    2025-05  14.507994     3.753101         9    12
29    2025-06  31.849130    11.171118        17    41
30    2025-07  41.956167    12.164468        31    43
31    2025-08  19.858055     4.329772         8    15
32    2025-09  67.445718    15.198787        28    62
33    2025-10  44.307074    14.390559        38    56
34    2025-11  46.819976    15.415378        34    60
35    2025-12  43.825329     6.168516        27    35
36    2026-01  21.999729     6.305147        15    33
37    2026-02  79.338685    20.556341        32    60
38    2026-03  47.863145    14.247743        26

,year,ventas_mm,utilidad_mm,facturas
0,2023,435.360263,135.814232,218
1,2024,593.824977,-52.583220,215
2,2025,535.289883,133.118609,290
3,2026,149.201559,41.109231,73


In [ ]:
# el error ocurre porque df_cali_analysis no tiene la columna "vendedor"

# primero debemos traer vendedor desde df_customers_enriched

# =========================================================
# AGREGAR VENDEDOR A df_cali_analysis
# =========================================================

df_vendedores = (
    df_customers_enriched[
        [
            "nit",
            "vendedor",
        ]
    ]
    .drop_duplicates(subset=["nit"])
    .copy()
)

df_cali_analysis = df_cali_analysis.merge(
    df_vendedores,
    on="nit",
    how="left"
)


# =========================================================
# NEGATIVE MARGIN REPORT
# toda la sucursal Cali
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# filtrar solo líneas con pérdida real
# ---------------------------------------------------------

negative_margin = (
    df_cali_analysis[
        df_cali_analysis["utilidad_total"] < 0
    ]
    .copy()
)

print("Rows con margen negativo:", negative_margin.shape[0])

# =========================================================
# TOP PERDIDAS POR LINEA
# =========================================================

top_negative_lines = (
    negative_margin[
        [
            "fecha",
            "razonsocial",
            "vendedor",
            "sku",
            "nombreproducto",
            "sufijo",
            "cantidad",
            "venta_total",
            "costo_total",
            "utilidad_total",
            "margen_pct",
            "numero",
        ]
    ]
    .sort_values(
        "utilidad_total",
        ascending=True
    )
    .copy()
)

# pasar a millones COP
money_cols = [
    "venta_total",
    "costo_total",
    "utilidad_total",
]

for col in money_cols:
    top_negative_lines[col] = (
        top_negative_lines[col] / 1_000_000
    ).round(2)

top_negative_lines["margen_pct"] = (
    top_negative_lines["margen_pct"] * 100
).round(1)

print("\nTOP PERDIDAS POR LINEA")
print(top_negative_lines.head(50))

# =========================================================
# TOP PERDIDAS AGRUPADAS POR CLIENTE + SKU
# =========================================================

top_negative_sku = (
    negative_margin
    .groupby(
        [
            "razonsocial",
            "vendedor",
            "sku",
            "nombreproducto",
            "sufijo",
        ],
        as_index=False
    )
    .agg(
        ventas_total=("venta_total", "sum"),
        costos_total=("costo_total", "sum"),
        utilidad_total=("utilidad_total", "sum"),
        cantidad_total=("cantidad", "sum"),
        facturas=("numero", "nunique"),
        ultima_venta=("fecha", "max"),
    )
)

top_negative_sku["margen_pct"] = np.where(
    top_negative_sku["ventas_total"] > 0,
    top_negative_sku["utilidad_total"] /
    top_negative_sku["ventas_total"],
    0
)

top_negative_sku = (
    top_negative_sku
    .sort_values(
        "utilidad_total",
        ascending=True
    )
    .copy()
)

for col in [
    "ventas_total",
    "costos_total",
    "utilidad_total",
]:
    top_negative_sku[col] = (
        top_negative_sku[col] / 1_000_000
    ).round(2)

top_negative_sku["margen_pct"] = (
    top_negative_sku["margen_pct"] * 100
).round(1)

print("\nTOP PERDIDAS POR CLIENTE + SKU")
print(top_negative_sku.head(50))

# =========================================================
# TOP CLIENTES CON MAYOR PERDIDA
# =========================================================

top_negative_clients = (
    negative_margin
    .groupby(
        [
            "razonsocial",
            "vendedor",
            "actividad_economica",
        ],
        as_index=False
    )
    .agg(
        ventas_total=("venta_total", "sum"),
        costos_total=("costo_total", "sum"),
        utilidad_total=("utilidad_total", "sum"),
        skus=("sku", "nunique"),
        facturas=("numero", "nunique"),
        ultima_venta=("fecha", "max"),
    )
)

top_negative_clients["margen_pct"] = np.where(
    top_negative_clients["ventas_total"] > 0,
    top_negative_clients["utilidad_total"] /
    top_negative_clients["ventas_total"],
    0
)

top_negative_clients = (
    top_negative_clients
    .sort_values(
        "utilidad_total",
        ascending=True
    )
    .copy()
)

for col in [
    "ventas_total",
    "costos_total",
    "utilidad_total",
]:
    top_negative_clients[col] = (
        top_negative_clients[col] / 1_000_000
    ).round(2)

top_negative_clients["margen_pct"] = (
    top_negative_clients["margen_pct"] * 100
).round(1)

print("\nTOP CLIENTES CON MAYOR PERDIDA")
print(top_negative_clients.head(30))

Rows con margen negativo: 1109

TOP PERDIDAS POR LINEA
           fecha                                        razonsocial  \
7467  2024-05-03                              CARTONES AMERICA S.A.   
7469  2024-05-03                              CARTONES AMERICA S.A.   
33416 2024-10-02                              CARTONES AMERICA S.A.   
52500 2025-11-07                               TERMOEMCALI I SA ESP   
46742 2025-07-04                            HARINERA DEL VALLE S.A.   
36126 2024-11-27                           INGENIO PROVIDENCIA S.A.   
53054 2025-11-20  CENIT TRANSPORTE Y LOGISTICA DE HIDROCARBUROS SAS   
19140 2023-04-11                            INGENIO DEL CAUCA S.A.S   
22802 2023-06-20  PRIME TERMOVALLE S.A.S. EMPRESA DE SERVICIOS P...   
7150  2024-04-26                            HARINERA DEL VALLE S.A.   
58450 2026-03-21                           INGENIO PROVIDENCIA S.A.   
32817 2023-12-29                     MYM BOBINADOS INDUSTRIALES SAS   
37144 2024-12-16      

In [ ]:
# =========================================================
# 1) PERDIDA COMERCIAL REAL
# venta positiva + utilidad negativa
# =========================================================

negative_margin_comercial = (
    df_cali_analysis[
        (df_cali_analysis["venta_total"] > 0) &
        (df_cali_analysis["utilidad_total"] < 0)
    ]
    .copy()
)

print("Rows con pérdida comercial real:", negative_margin_comercial.shape[0])

top_negative_comercial = (
    negative_margin_comercial
    .groupby(
        [
            "razonsocial",
            "vendedor",
            "sku",
            "nombreproducto",
            "sufijo",
        ],
        as_index=False
    )
    .agg(
        ventas_total=("venta_total", "sum"),
        costos_total=("costo_total", "sum"),
        utilidad_total=("utilidad_total", "sum"),
        cantidad_total=("cantidad", "sum"),
        facturas=("numero", "nunique"),
        ultima_venta=("fecha", "max"),
    )
)

top_negative_comercial["margen_pct"] = (
    top_negative_comercial["utilidad_total"] /
    top_negative_comercial["ventas_total"]
)

top_negative_comercial = top_negative_comercial.sort_values(
    "utilidad_total",
    ascending=True
).copy()

for col in ["ventas_total", "costos_total", "utilidad_total"]:
    top_negative_comercial[col] = (top_negative_comercial[col] / 1_000_000).round(2)

top_negative_comercial["margen_pct"] = (top_negative_comercial["margen_pct"] * 100).round(1)

top_negative_comercial.head(50)

Rows con pérdida comercial real: 234


,razonsocial,vendedor,sku,nombreproducto,sufijo,ventas_total,costos_total,utilidad_total,cantidad_total,facturas,ultima_venta,margen_pct
23,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,239/560 CAK/W33SKF,RODAMIENTO DE RODILLOS ESFERICOS,SKF,55.12,75.64,-20.51,1.0,1,2024-10-02,-37.2
77,CARVAJAL PULPA Y PAPEL S.A.,NUBIA ANDREA JIMENEZ,SNL 524-620/TSN 524 LLH,SOPORTE CON SELLOS TSN,LH,2.74,3.52,-0.79,2.0,2,2024-10-11,-28.7
168,SIDERURGICA DE OCCIDENTE S.A.S,JAIRO DAVID VERA,COP.ATL125MX2SKF-LH,SELLOS PARA SOPORTE COOPER X 2,SKF-LH,0.73,1.42,-0.69,1.0,1,2025-04-24,-93.9
160,RODAMIENTOS INDUSTRIALES DE OCCIDENTE SA,ALMACEN CALI -UNO-,TS 2806+3000LTHK,TORNILLO DE BOLAS,THK,2.79,3.35,-0.55,1.0,1,2026-03-26,-19.8
90,INGENIO PROVIDENCIA S.A.,JAIRO DAVID VERA,23156 CCK/C3W33SKF,RODAMIENTO DE RODILLOS ESFERICOS,SKF,27.21,27.76,-0.55,2.0,1,2025-09-22,-2.0
155,PRODUCTORA DE CONFITES Y CHICLET´S MAC DULCES...,NUBIA ANDREA JIMENEZ,NKI 85/36-XLINA,RODAMIENTO DE AGUJAS,INA,2.35,2.81,-0.46,5.0,1,2023-05-26,-19.5
37,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,7317 BECBJSKF,RODAMIENTO DE CONTACTO ANGULAR,SKF,2.59,2.85,-0.26,2.0,1,2026-02-02,-10.0
88,HARINERA DEL VALLE S.A.,JAIRO DAVID VERA,203 KRR2FAF,RODAMIENTO RIGIDO DE BOLAS,FAF,0.36,0.59,-0.23,8.0,1,2023-05-10,-64.4
55,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,KM 32SKF,TUERCA DE FIJACION,SKF,2.12,2.31,-0.20,6.0,3,2025-04-04,-9.4
21,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,23124 CC/C3W33SKF,RODAMIENTO DE RODILLOS ESFERICOS,SKF,4.06,4.25,-0.20,2.0,1,2023-12-22,-4.9


In [ ]:
negative_margin_clients_review = (
    top_negative_comercial
    .groupby(["razonsocial", "vendedor"], as_index=False)
    .agg(
        perdidas_mm=("utilidad_total", "sum"),
        ventas_mm=("ventas_total", "sum"),
        skus_negativos=("sku", "nunique"),
        facturas_negativas=("facturas", "sum"),
        ultima_venta=("ultima_venta", "max"),
    )
    .sort_values("perdidas_mm")
    .copy()
)

negative_margin_clients_review["margen_pct"] = (
    negative_margin_clients_review["perdidas_mm"] /
    negative_margin_clients_review["ventas_mm"]
).round(4)

negative_margin_clients_review["margen_pct"] = (
    negative_margin_clients_review["margen_pct"] * 100
).round(1)

negative_margin_clients_review.head(20)

,razonsocial,vendedor,perdidas_mm,ventas_mm,skus_negativos,facturas_negativas,ultima_venta,margen_pct
7,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,-22.72,96.05,57,91,2026-03-05,-23.6
24,PAPELES NACIONALES S.A.S,JAIRO DAVID VERA,-1.02,30.73,33,46,2026-03-19,-3.3
9,CARVAJAL PULPA Y PAPEL S.A.,NUBIA ANDREA JIMENEZ,-0.79,3.85,2,3,2024-10-11,-20.5
34,SIDERURGICA DE OCCIDENTE S.A.S,JAIRO DAVID VERA,-0.69,0.73,1,1,2025-04-24,-94.5
29,RODAMIENTOS INDUSTRIALES DE OCCIDENTE SA,ALMACEN CALI -UNO-,-0.59,2.83,3,3,2026-03-26,-20.8
18,INGENIO PROVIDENCIA S.A.,JAIRO DAVID VERA,-0.55,27.21,1,1,2025-09-22,-2.0
26,PRODUCTORA DE CONFITES Y CHICLET´S MAC DULCES...,NUBIA ANDREA JIMENEZ,-0.46,2.35,1,1,2023-05-26,-19.6
23,NESTLE DE COLOMBIA S.A.,JEISMAN HOLGUIN,-0.38,4.54,26,30,2025-06-24,-8.4
16,HARINERA DEL VALLE S.A.,JAIRO DAVID VERA,-0.23,0.36,1,1,2023-05-10,-63.9
14,FEDERACION NACIONAL DE CAFETEROS DE COLOMBIA,YEISSON ANDRES RENTERIA MOSQUERA,-0.19,1.39,1,1,2025-03-05,-13.7


In [ ]:
revision_negativos = top_negative_comercial.copy()

revision_negativos["hipotesis"] = ""
revision_negativos["accion"] = ""
revision_negativos["responsable_revision"] = ""
revision_negativos["fecha_revision"] = pd.NaT
revision_negativos["status_revision"] = "Pendiente"

revision_negativos = revision_negativos[
    [
        "razonsocial",
        "vendedor",
        "sku",
        "nombreproducto",
        "ventas_total",
        "costos_total",
        "utilidad_total",
        "margen_pct",
        "facturas",
        "ultima_venta",
        "hipotesis",
        "accion",
        "responsable_revision",
        "fecha_revision",
        "status_revision",
    ]
].copy()

revision_negativos.head(20)

,razonsocial,vendedor,sku,nombreproducto,ventas_total,costos_total,utilidad_total,margen_pct,facturas,ultima_venta,hipotesis,accion,responsable_revision,fecha_revision,status_revision
23,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,239/560 CAK/W33SKF,RODAMIENTO DE RODILLOS ESFERICOS,55.12,75.64,-20.51,-37.2,1,2024-10-02,,,,NaT,Pendiente
77,CARVAJAL PULPA Y PAPEL S.A.,NUBIA ANDREA JIMENEZ,SNL 524-620/TSN 524 LLH,SOPORTE CON SELLOS TSN,2.74,3.52,-0.79,-28.7,2,2024-10-11,,,,NaT,Pendiente
168,SIDERURGICA DE OCCIDENTE S.A.S,JAIRO DAVID VERA,COP.ATL125MX2SKF-LH,SELLOS PARA SOPORTE COOPER X 2,0.73,1.42,-0.69,-93.9,1,2025-04-24,,,,NaT,Pendiente
160,RODAMIENTOS INDUSTRIALES DE OCCIDENTE SA,ALMACEN CALI -UNO-,TS 2806+3000LTHK,TORNILLO DE BOLAS,2.79,3.35,-0.55,-19.8,1,2026-03-26,,,,NaT,Pendiente
90,INGENIO PROVIDENCIA S.A.,JAIRO DAVID VERA,23156 CCK/C3W33SKF,RODAMIENTO DE RODILLOS ESFERICOS,27.21,27.76,-0.55,-2.0,1,2025-09-22,,,,NaT,Pendiente
155,PRODUCTORA DE CONFITES Y CHICLET´S MAC DULCES...,NUBIA ANDREA JIMENEZ,NKI 85/36-XLINA,RODAMIENTO DE AGUJAS,2.35,2.81,-0.46,-19.5,1,2023-05-26,,,,NaT,Pendiente
37,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,7317 BECBJSKF,RODAMIENTO DE CONTACTO ANGULAR,2.59,2.85,-0.26,-10.0,1,2026-02-02,,,,NaT,Pendiente
88,HARINERA DEL VALLE S.A.,JAIRO DAVID VERA,203 KRR2FAF,RODAMIENTO RIGIDO DE BOLAS,0.36,0.59,-0.23,-64.4,1,2023-05-10,,,,NaT,Pendiente
55,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,KM 32SKF,TUERCA DE FIJACION,2.12,2.31,-0.20,-9.4,3,2025-04-04,,,,NaT,Pendiente
21,CARTONES AMERICA S.A.,JEISMAN HOLGUIN,23124 CC/C3W33SKF,RODAMIENTO DE RODILLOS ESFERICOS,4.06,4.25,-0.20,-4.9,1,2023-12-22,,,,NaT,Pendiente


In [ ]:
# =========================================================
# NEGATIVE MARGIN BY VENDEDOR
# donde realmente se concentra el problema
# =========================================================

negative_by_vendedor = (
    top_negative_comercial
    .groupby(
        ["vendedor"],
        as_index=False
    )
    .agg(
        perdidas_mm=("utilidad_total", "sum"),
        ventas_mm=("ventas_total", "sum"),
        clientes_afectados=("razonsocial", "nunique"),
        skus_negativos=("sku", "nunique"),
        facturas_negativas=("facturas", "sum"),
        ultima_revision=("ultima_venta", "max"),
    )
    .copy()
)

# ---------------------------------------------------------
# margen consolidado
# ---------------------------------------------------------

negative_by_vendedor["margen_pct"] = (
    negative_by_vendedor["perdidas_mm"] /
    negative_by_vendedor["ventas_mm"]
)

negative_by_vendedor["margen_pct"] = (
    negative_by_vendedor["margen_pct"] * 100
).round(1)

# ---------------------------------------------------------
# ordenar por mayor pérdida
# ---------------------------------------------------------

negative_by_vendedor = (
    negative_by_vendedor
    .sort_values(
        "perdidas_mm",
        ascending=True
    )
    .reset_index(drop=True)
)

negative_by_vendedor["rank_perdida"] = (
    negative_by_vendedor.index + 1
)

# ---------------------------------------------------------
# columnas finales
# ---------------------------------------------------------

negative_by_vendedor = (
    negative_by_vendedor[
        [
            "rank_perdida",
            "vendedor",
            "perdidas_mm",
            "ventas_mm",
            "margen_pct",
            "clientes_afectados",
            "skus_negativos",
            "facturas_negativas",
            "ultima_revision",
        ]
    ]
)

print("\nNEGATIVE MARGIN BY VENDEDOR")
negative_by_vendedor


NEGATIVE MARGIN BY VENDEDOR


,rank_perdida,vendedor,perdidas_mm,ventas_mm,margen_pct,clientes_afectados,skus_negativos,facturas_negativas,ultima_revision
0,1,JEISMAN HOLGUIN,-23.33,103.40,-22.6,6,88,126,2026-03-20
1,2,JAIRO DAVID VERA,-2.79,71.08,-3.9,7,40,55,2026-03-19
2,3,NUBIA ANDREA JIMENEZ,-1.26,6.61,-19.1,6,7,8,2026-03-05
3,4,ALMACEN CALI -UNO-,-0.99,6.19,-16.0,12,19,22,2026-03-31
4,5,YEISSON ANDRES RENTERIA MOSQUERA,-0.19,1.39,-13.7,1,1,1,2025-03-05
5,6,ALMACEN,-0.06,0.51,-11.8,3,3,3,2026-03-24
6,7,FABIO NELSON VALENCIA,-0.05,0.09,-55.6,1,1,4,2024-01-13
7,8,JHON ALEXANDER PINZON,-0.01,0.93,-1.1,1,10,12,2024-04-03


In [ ]:
print(df_sales["fecha"].head(20).tolist())
print(df_sales["fecha"].sample(20).tolist())

['09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00', '09/02/24 00:00:00']
['09/09/24 00:00:00', '10/31/25 00:00:00', '03/26/26 00:00:00', '08/17/24 00:00:00', '10/11/23 00:00:00', '11/24/23 00:00:00', '01/12/23 00:00:00', '10/23/25 00:00:00', '09/12/23 00:00:00', '04/10/24 00:00:00', '12/17/25 00:00:00', '10/01/25 00:00:00', '05/19/25 00:00:00', '05/28/25 00:00:00', '02/20/23 00:00:00', '02/18/26 00:00:00', '12/09/25 00:00:00', '05/10/23 00:00:00', '04/02/25 00:00:00', '08/17/23 00:00:00']


In [ ]:
# Ver valores únicos de Estado
print("Valores únicos en Estado:")
print(df_crm["Estado"].dropna().unique())

print("\n" + "="*50 + "\n")

# Ver valores únicos de Etapa
print("Valores únicos en Etapa:")
print(df_crm["Etapa"].dropna().unique())

Valores únicos en Estado:
['Abierto' 'Cancelado' 'Realizado']


Valores únicos en Etapa:
['Contacto' 'Propuesta o Cotizacion' 'Negociacion' 'Ganado'
 'Inicio Relacion Comercial']


In [ ]:
df_crm["Probabilidad"].unique()

array([  0, 100,  50,  75,   5])

In [ ]:
df_crm.groupby(["Estado", "Etapa"])["Probabilidad"].unique()

Estado     Etapa                    
Abierto    Contacto                        [0, 5]
           Negociacion                       [75]
           Propuesta o Cotizacion            [50]
Cancelado  Contacto                           [0]
           Inicio Relacion Comercial          [0]
           Negociacion                        [0]
           Propuesta o Cotizacion         [0, 50]
Realizado  Contacto                         [100]
           Ganado                           [100]
           Negociacion                      [100]
           Propuesta o Cotizacion       [100, 50]
Name: Probabilidad, dtype: object

In [ ]:
# -----------------------------
# Limpieza CRM
# -----------------------------

df_crm["Fecha"] = pd.to_datetime(df_crm["Fecha"], errors="coerce")
df_crm["Fecha Cierre"] = pd.to_datetime(df_crm["Fecha Cierre"], errors="coerce")

df_crm["Ingreso"] = pd.to_numeric(df_crm["Ingreso"], errors="coerce").fillna(0)
df_crm["Probabilidad"] = pd.to_numeric(df_crm["Probabilidad"], errors="coerce").fillna(0)

# Probabilidad limpia basada en Estado
df_crm["Probabilidad_clean"] = df_crm["Probabilidad"]

df_crm.loc[
    df_crm["Estado"] == "Cancelado",
    "Probabilidad_clean"
] = 0

df_crm.loc[
    df_crm["Estado"] == "Realizado",
    "Probabilidad_clean"
] = 100

# Ingreso ponderado para forecast
df_crm["Ingreso_ponderado"] = (
    df_crm["Ingreso"] *
    df_crm["Probabilidad_clean"] / 100
)

# Aging comercial
today = pd.Timestamp.today().normalize()

df_crm["Dias_abierto"] = (
    today - df_crm["Fecha"]
).dt.days

# Oportunidades vencidas:
# fecha cierre menor a hoy y todavía abiertas
df_crm["Oportunidad_vencida"] = (
    (df_crm["Estado"] == "Abierto") &
    (df_crm["Fecha Cierre"] < today)
)

print("CRM limpio listo")
print(df_crm.shape)

CRM limpio listo
(189, 20)


In [ ]:
# -----------------------------
# KPIs principales CRM
# -----------------------------

kpis_crm = {
    "oportunidades_totales": len(df_crm),
    "oportunidades_abiertas": (df_crm["Estado"] == "Abierto").sum(),
    "oportunidades_realizadas": (df_crm["Estado"] == "Realizado").sum(),
    "oportunidades_canceladas": (df_crm["Estado"] == "Cancelado").sum(),
    "pipeline_total_abierto": df_crm.loc[df_crm["Estado"] == "Abierto", "Ingreso"].sum(),
    "pipeline_ponderado_abierto": df_crm.loc[df_crm["Estado"] == "Abierto", "Ingreso_ponderado"].sum(),
    "ventas_realizadas_crm": df_crm.loc[df_crm["Estado"] == "Realizado", "Ingreso"].sum(),
    "valor_cancelado": df_crm.loc[df_crm["Estado"] == "Cancelado", "Ingreso"].sum(),
    "oportunidades_vencidas": df_crm["Oportunidad_vencida"].sum(),
    "aging_promedio_abiertas": df_crm.loc[df_crm["Estado"] == "Abierto", "Dias_abierto"].mean(),
}

for k, v in kpis_crm.items():
    print(f"{k}: {v:,.2f}" if isinstance(v, float) else f"{k}: {v:,}")

oportunidades_totales: 189
oportunidades_abiertas: 66
oportunidades_realizadas: 55
oportunidades_canceladas: 68
pipeline_total_abierto: 2,102,908,758
pipeline_ponderado_abierto: 813,060,514.00
ventas_realizadas_crm: 561,169,910
valor_cancelado: 1,995,469,055
oportunidades_vencidas: 60
aging_promedio_abiertas: 27.33


In [ ]:
# 1. Estado
print(df_crm["Estado"].value_counts(dropna=False))

print("\n" + "="*60 + "\n")

# 2. Etapa
print(df_crm["Etapa"].value_counts(dropna=False))

print("\n" + "="*60 + "\n")

# 3. Pipeline por vendedor
pipeline_vendedor = (
    df_crm[df_crm["Estado"] == "Abierto"]
    .groupby("Vendedor", dropna=False)
    .agg(
        oportunidades=("ID", "count"),
        pipeline_total=("Ingreso", "sum"),
        pipeline_ponderado=("Ingreso_ponderado", "sum"),
        vencidas=("Oportunidad_vencida", "sum"),
    )
    .sort_values("pipeline_total", ascending=False)
)

print(pipeline_vendedor)

print("\n" + "="*60 + "\n")

# 4. Oportunidades abiertas más grandes
top_abiertas = (
    df_crm[df_crm["Estado"] == "Abierto"]
    .sort_values("Ingreso", ascending=False)
    [[
        "Fecha", "Fecha Cierre", "Nombre empresa", "Documento",
        "Ingreso", "Probabilidad_clean", "Etapa", "Vendedor",
        "Dias_abierto", "Oportunidad_vencida"
    ]]
    .head(20)
)

print(top_abiertas)

Estado
Cancelado    68
Abierto      66
Realizado    55
Name: count, dtype: int64


Etapa
Contacto                     115
Propuesta o Cotizacion        42
Negociacion                   20
Ganado                        11
Inicio Relacion Comercial      1
Name: count, dtype: int64


                                  oportunidades  pipeline_total  \
Vendedor                                                          
NUBIA ANDREA JIMENEZ                         18       621994795   
ALMACEN                                       7       541874280   
JEISMAN HOLGUIN                               1       339353475   
EDWIN ARBELAEZ                                6       229060265   
JHON ALEXANDER PINZON                        13       203461740   
JAIRO DAVID VERA                             10        85293045   
FERNEY ORDOÑEZ                                3        44153238   
JUAN CARLOS BENAVIDES                         4        34917885   
WHATSAPP MARÍA                                1 

KeyError: "['Nombre empresa'] not in index"

In [ ]:
# Primero revisemos el nombre real de las columnas

print(df_crm.columns.tolist())

['ID', 'Fecha', 'Prioridad', 'Oportunidad', 'Nombre empresa ', 'Ingreso', 'Documento', 'Probabilidad', 'Fecha Cierre', 'Vendedor', 'Creado por', 'Estado', 'Etapa', 'Teléfono', 'Movil', 'Ciudad', 'Probabilidad_clean', 'Ingreso_ponderado', 'Dias_abierto', 'Oportunidad_vencida']


In [ ]:
# limpiar nombres de columnas
df_crm.columns = df_crm.columns.str.strip()

print(df_crm.columns.tolist())

['ID', 'Fecha', 'Prioridad', 'Oportunidad', 'Nombre empresa', 'Ingreso', 'Documento', 'Probabilidad', 'Fecha Cierre', 'Vendedor', 'Creado por', 'Estado', 'Etapa', 'Teléfono', 'Movil', 'Ciudad', 'Probabilidad_clean', 'Ingreso_ponderado', 'Dias_abierto', 'Oportunidad_vencida']


In [ ]:
top_abiertas = (
    df_crm[df_crm["Estado"] == "Abierto"]
    .sort_values("Ingreso", ascending=False)
    [[
        "Fecha", "Fecha Cierre", "Nombre empresa", "Documento",
        "Ingreso", "Probabilidad_clean", "Etapa", "Vendedor",
        "Dias_abierto", "Oportunidad_vencida"
    ]]
    .head(20)
)

top_abiertas

,Fecha,Fecha Cierre,Nombre empresa,Documento,Ingreso,Probabilidad_clean,Etapa,Vendedor,Dias_abierto,Oportunidad_vencida
42,2026-04-14 11:22:44.121,2026-04-16,POLLOS EL BUCANERO S.A.,NaN,339353475,50,Propuesta o Cotizacion,JEISMAN HOLGUIN,8,True
112,2026-03-13 11:46:25.793,2026-03-19,CARTON DE COLOMBIA S.A.,2-PC1-474,241413250,0,Contacto,NUBIA ANDREA JIMENEZ,40,True
18,2026-04-16 11:49:57.147,2026-04-20,TWM S.A.S,1-PCT-1609,213401640,50,Propuesta o Cotizacion,ALMACEN,6,True
119,2026-03-12 11:04:03.819,2026-03-18,INEMEC S.A.S.,1-PCT-1400,202347350,75,Negociacion,ALMACEN,41,True
149,2026-02-27 11:25:36.533,2026-03-03,CARTON DE COLOMBIA S.A.,2-PC1-442,123452560,50,Propuesta o Cotizacion,NUBIA ANDREA JIMENEZ,54,True
95,2026-03-18 11:13:40.845,2026-03-30,PAPELES Y CARTONES SA PAPELSA,1-PCT-1451,102881220,0,Contacto,EDWIN ARBELAEZ,35,True
16,2026-04-16 17:36:54.298,2026-04-16,TERMOTASAJERO DOS S.A. E.S.P.,1-PCT-1624,94134330,75,Negociacion,JHON ALEXANDER PINZON,6,True
152,2026-02-25 11:01:47.988,2026-02-27,MANUELITA S.A.,1-PCT-1310,67016205,50,Propuesta o Cotizacion,EDWIN ARBELAEZ,56,True
60,2026-04-06 15:00:18.566,2026-04-10,COLOMBINA S.A.,2-PC1-497,65731050,0,Contacto,NUBIA ANDREA JIMENEZ,16,True
53,2026-04-09 12:41:50.157,2026-04-16,COPOWER LIMITADA,1-PCT-1565,59595480,75,Negociacion,ALMACEN,13,True


In [ ]:
# Ver lista de vendedores únicos en CRM

vendedores = (
    df_crm["Vendedor"]
    .dropna()
    .astype(str)
    .str.strip()
    .sort_values()
    .unique()
)

print("Vendedores encontrados:\n")

for vendedor in vendedores:
    print(vendedor)

Vendedores encontrados:

ADBEIRO CUESTA
ALMACEN
EDWIN ARBELAEZ
FERNEY ORDOÑEZ
FERNEY RINCON
GERMAN BELTRAN
JAIRO DAVID VERA
JEISMAN HOLGUIN
JHON ALEXANDER PINZON
JOHAN SEBASTIAN CORDOBA GAMBOA
JOHN FREDY MORENO
JUAN CARLOS BENAVIDES
MARIA ELSA SIERRA FERNANDEZ
MAURICIO ALBERTO SUAREZ
NUBIA ANDREA JIMENEZ
WHATSAPP MARÍA
YEISSON ANDRES RENTERIA MOSQUERA


In [ ]:
# -----------------------------
# Filtrar CRM solo vendedores Cali
# -----------------------------

vendedores_cali = [
    "JAIRO DAVID VERA",
    "JEISMAN HOLGUIN",
    "YEISSON ANDRES RENTERIA MOSQUERA",
]

df_crm_cali = (
    df_crm[
        df_crm["Vendedor"]
        .astype(str)
        .str.strip()
        .isin(vendedores_cali)
    ]
    .copy()
)

print("CRM Cali listo")
print(df_crm_cali.shape)

print("\nVendedores incluidos:")
print(df_crm_cali["Vendedor"].value_counts())


CRM Cali listo
(52, 20)

Vendedores incluidos:
Vendedor
JAIRO DAVID VERA                    45
YEISSON ANDRES RENTERIA MOSQUERA     4
JEISMAN HOLGUIN                      3
Name: count, dtype: int64


In [ ]:
# -----------------------------
# KPIs principales CRM Cali
# -----------------------------

kpis_crm_cali = {
    "oportunidades_totales": len(df_crm_cali),
    "oportunidades_abiertas": (df_crm_cali["Estado"] == "Abierto").sum(),
    "oportunidades_realizadas": (df_crm_cali["Estado"] == "Realizado").sum(),
    "oportunidades_canceladas": (df_crm_cali["Estado"] == "Cancelado").sum(),
    "pipeline_total_abierto": df_crm_cali.loc[
        df_crm_cali["Estado"] == "Abierto",
        "Ingreso"
    ].sum(),
    "pipeline_ponderado_abierto": df_crm_cali.loc[
        df_crm_cali["Estado"] == "Abierto",
        "Ingreso_ponderado"
    ].sum(),
    "ventas_realizadas_crm": df_crm_cali.loc[
        df_crm_cali["Estado"] == "Realizado",
        "Ingreso"
    ].sum(),
    "valor_cancelado": df_crm_cali.loc[
        df_crm_cali["Estado"] == "Cancelado",
        "Ingreso"
    ].sum(),
    "oportunidades_vencidas": df_crm_cali["Oportunidad_vencida"].sum(),
    "aging_promedio_abiertas": df_crm_cali.loc[
        df_crm_cali["Estado"] == "Abierto",
        "Dias_abierto"
    ].mean(),
}

for k, v in kpis_crm_cali.items():
    print(f"{k}: {v:,.2f}" if isinstance(v, float) else f"{k}: {v:,}")

NameError: name 'df_crm_cali' is not defined

In [ ]:
df_customers.head()

,nit,dv,razonsocial,escliente,cliente_credito,direccion1,ciudad,telefono1,movil,email,emailfe,vendedor,cupocreditocc,plazopagocc,idciiu,ID
0,890914711,0,SEALCO S.A.,S,S,CALLE 9 N. 68-65,BOGOTA D.C.,2606996,,,auxalmacen@cisealco.com,ALMACEN,10000000,60,2599,1
1,444444830,,SUMINISTROS INDUSTRIALES NAPA S.A. DE C.V,S,S,"Juan Ponce de León 2890, Colonia 18 de Marzo. ...",Ciudad de Mexico,33 3550 1468,52 33 1813 2025,,oscar.nares@napasa.com.mx,ALMACEN,0,30,4752,2
2,901636404,0,MAXXIFLEX D&P SAS,S,S,CL 16 SUR NO 24I-85,Bogota D.C.,,3124620292,,MAXXIFLEX202@HOTMAIL.COM,ALMACEN,0,30,1523,3
3,11257332,,GALLO RODRIGUEZ TOBIAS FERNANDO,S,S,CR 69 No 14-41,Bogota D.C.,3133508362,,,pulpfriccion@gmail.com,ALMACEN,0,30,4752,4
4,1023870861,,SOLORZANO RODRIGUEZ JHONATAN FERNANDO,S,S,CR 12 C N. 37-24 SUR,BOGOTA D.C.,2068076,,fernando-solorzano@lugohermanos.com,fernando-solorzano@lugohermanos.com,ALMACEN,0,60,4752,5


In [ ]:
top_cuentas_cali.head()

NameError: name 'top_cuentas_cali' is not defined